In [13]:
from pyspark.sql.types import *
import pyspark.sql.functions as F
# import the Databricks Connect library that bridges VS Code to your Databricks workspace
from databricks.connect import DatabricksSession
# create the spark session connected to your Databricks serveless compute 
spark = DatabricksSession.builder.getOrCreate()

###

# I. With StringType() and ArrayType()

### I.1. StringType()

* In this example: stringtype will be written as a dictionary

In [14]:
schema= StructType([
    StructField("CustomerID", IntegerType(), False),
    StructField("json_data", StringType(), True)
])

In [15]:
# get data as a list of rows
data= [
    Row(CustomerID= 1,
        json_data= {"name": "ALice", "age": 25}),
    
    Row(CustomerID= 2,
        json_data= {"name": "Bob", "age": 30})
]

In [16]:
df= spark.createDataFrame(data, schema)

In [17]:
df.printSchema()

root
 |-- CustomerID: integer (nullable = false)
 |-- json_data: string (nullable = true)



In [18]:
df.show(truncate=False)

+----------+----------------------------+
|CustomerID|json_data                   |
+----------+----------------------------+
|1         |{'name': 'ALice', 'age': 25}|
|2         |{'name': 'Bob', 'age': 30}  |
+----------+----------------------------+



* Now we want to convert the column json_data into a nested structure StrucType() by using from_json(column, schema) function

In [19]:
schema_column= StructType([
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True)
])

In [20]:
df_new= df.select(
    "CustomerID",
    F.from_json("json_data", schema_column).alias("CustomerInfo")
)

In [21]:
df_new.show(truncate= False)

+----------+------------+
|CustomerID|CustomerInfo|
+----------+------------+
|1         |{ALice, 25} |
|2         |{Bob, 30}   |
+----------+------------+



In [22]:
df_new.printSchema()

root
 |-- CustomerID: integer (nullable = false)
 |-- CustomerInfo: struct (nullable = true)
 |    |-- name: string (nullable = true)
 |    |-- age: integer (nullable = true)

